# 🎯 NumPy: Zero to Master — A Guided Lab

Welcome! This is a complete, self-contained course. By the end you'll be able to use NumPy
fluently for real data work and understand *why* things work, not just *how*.

**How this lab works**
- Each **chapter** has: 📖 Theory → 🔬 Worked examples → ✏️ Your Turn exercises → ✅ Solutions.
- Run **every** code cell as you go (Shift+Enter). Don't just read — type and experiment.
- Try each ✏️ exercise *before* opening the ✅ solution beneath it.
- 🧠 boxes explain the mental model; ⚡ boxes are pro tips; ⚠️ boxes are common traps.

**What you'll master**
1. Why NumPy exists & the ndarray
2. Creating arrays (every method you'll actually use)
3. Array attributes, dtypes & memory
4. Indexing & slicing (incl. the tricky parts)
5. Boolean masking & fancy indexing
6. Vectorized math & universal functions (ufuncs)
7. Broadcasting (the concept that unlocks NumPy)
8. Aggregations & the `axis` parameter
9. Reshaping, stacking & splitting
10. Sorting, searching & set operations
11. Linear algebra essentials
12. Random numbers & simulation
13. 🏆 Capstone project: analyze a real dataset

Let's begin. First, the one import you'll use everywhere:


In [ ]:
import numpy as np
print("NumPy version:", np.__version__)

---
## Chapter 1 — Why NumPy? The ndarray

📖 **Theory.** Python lists are flexible but slow for math: each element is a full Python
object, and operations loop in the (slow) Python interpreter. NumPy stores data in a
compact, fixed-type block of memory (the **ndarray**) and runs operations in fast compiled
C code. This makes NumPy often 10–100× faster *and* more memory-efficient.

🧠 **Mental model.** A Python list is a shelf of labeled boxes that can each hold anything.
A NumPy array is a single ruler-straight strip of identical cells — the computer can zoom
through it because every cell is the same size and type.

Let's see the speed difference for ourselves.

In [ ]:
import time

size = 1_000_000
py_list = list(range(size))
np_array = np.arange(size)

# Square every element with a Python list
start = time.time()
_ = [x**2 for x in py_list]
list_time = time.time() - start

# Square every element with NumPy
start = time.time()
_ = np_array**2
numpy_time = time.time() - start

print(f"Python list: {list_time:.4f}s")
print(f"NumPy array: {numpy_time:.4f}s")
print(f"NumPy is ~{list_time/numpy_time:.0f}x faster here")

⚡ **Pro tip.** The speedup comes from *vectorization* — expressing an operation on the
whole array at once (`np_array**2`) instead of looping element by element. This is the single
most important habit in NumPy. We'll return to it constantly.

### ✏️ Your Turn 1.1
Create a Python list and a NumPy array both containing the numbers 0–9. Then produce a new
version where every number is multiplied by 3 — once using a list comprehension, once using
NumPy vectorization.

In [ ]:
# Your code here
py = None          # list 0-9
arr = None         # numpy array 0-9
py_times3 = None   # list comprehension * 3
arr_times3 = None  # vectorized * 3
print(py_times3, arr_times3)

✅ **Solution**
```python
py = list(range(10))
arr = np.arange(10)
py_times3 = [x*3 for x in py]
arr_times3 = arr * 3
```

---
## Chapter 2 — Creating Arrays

📖 **Theory.** There are many ways to make arrays. You'll reach for these constantly:

| Function | What it does |
|---|---|
| `np.array([...])` | from a Python list/tuple |
| `np.arange(start, stop, step)` | like `range`, but returns an array |
| `np.linspace(start, stop, num)` | `num` evenly-spaced points *including* both ends |
| `np.zeros(shape)` / `np.ones(shape)` | filled with 0s / 1s |
| `np.full(shape, value)` | filled with a constant |
| `np.eye(n)` | identity matrix |
| `np.random.rand(...)` | random floats in [0,1) |

🧠 **Mental model.** `arange` counts by *step size*; `linspace` counts by *number of points*.
Use `arange` when you know the spacing, `linspace` when you know how many you want.

In [ ]:
from_list = np.array([1, 2, 3, 4])
ranged    = np.arange(0, 10, 2)          # 0,2,4,6,8  (stop is exclusive)
spaced    = np.linspace(0, 1, 5)         # 0, 0.25, 0.5, 0.75, 1  (stop included)
zeros     = np.zeros((2, 3))             # 2x3 of zeros
ones      = np.ones((2, 3))
constant  = np.full((2, 2), 7)
identity  = np.eye(3)

print("from_list:", from_list)
print("ranged:   ", ranged)
print("spaced:   ", spaced)
print("zeros:\n", zeros)
print("identity:\n", identity)

⚠️ **Common trap.** `np.arange`'s `stop` is *exclusive* (like Python's `range`), but
`np.linspace`'s `stop` is *inclusive* by default. Mixing these up is a classic off-by-one bug.

### ✏️ Your Turn 2.1
Create the following, each in one line:
1. `temps` — 7 evenly spaced values from 0 to 30 (inclusive)
2. `grid` — a 3×3 array full of the value 5
3. `evens` — all even numbers from 10 up to (but not including) 30

In [ ]:
temps = None
grid = None
evens = None
print(temps); print(grid); print(evens)

✅ **Solution**
```python
temps = np.linspace(0, 30, 7)
grid  = np.full((3, 3), 5)
evens = np.arange(10, 30, 2)
```

---
## Chapter 3 — Attributes, dtypes & Memory

📖 **Theory.** Every array carries metadata you'll inspect constantly:
- `.shape` — the size along each dimension (a tuple)
- `.ndim` — number of dimensions
- `.size` — total number of elements
- `.dtype` — the data type of the elements (e.g. `int64`, `float64`, `bool`)

🧠 **Mental model.** `shape` answers "how is it laid out?", `dtype` answers "what kind of
numbers live inside?". Getting `dtype` wrong (e.g. integers when you needed floats) silently
corrupts results — integer division truncates, integer arrays can't hold NaN.

In [ ]:
a = np.array([[1, 2, 3], [4, 5, 6]])
print("shape:", a.shape)     # (2, 3)
print("ndim: ", a.ndim)      # 2
print("size: ", a.size)      # 6
print("dtype:", a.dtype)     # int64

# Change dtype explicitly
floats = a.astype(np.float64)
print("as float dtype:", floats.dtype)

# dtype affects behavior!
int_arr = np.array([1, 2, 3])
print("int division:", int_arr / 2)     # becomes float automatically here
ints = np.array([7, 8, 9])
ints[0] = 3.9                             # gets truncated to 3 -- integer array!
print("truncated assignment:", ints)

⚠️ **Common trap.** Assigning a float into an integer array silently truncates it. If you
need decimals, create the array as float from the start (`np.array([1.0, 2.0])` or
`dtype=float`).

### ✏️ Your Turn 3.1
Create a 2×4 array of your choice. Print its shape, ndim, size, and dtype. Then create a
float version of it and confirm the dtype changed.

In [ ]:
arr = None
# print shape, ndim, size, dtype, then make a float copy


✅ **Solution**
```python
arr = np.array([[1,2,3,4],[5,6,7,8]])
print(arr.shape, arr.ndim, arr.size, arr.dtype)
arr_float = arr.astype(float)
print(arr_float.dtype)   # float64
```

---
## Chapter 4 — Indexing & Slicing

📖 **Theory.** Access elements with `[]`. For 2D arrays use `[row, col]`. Slices use
`start:stop:step` (stop exclusive), and you can slice each dimension independently.

🧠 **Mental model.** Think `[rows, columns]`. A bare `:` means "all of this dimension".

In [ ]:
a = np.array([[10, 11, 12, 13],
              [20, 21, 22, 23],
              [30, 31, 32, 33]])

print("single element [1,2]:", a[1, 2])      # 22
print("whole row 0:         ", a[0])          # [10 11 12 13]
print("whole column 1:      ", a[:, 1])       # [11 21 31]
print("sub-block rows0-1,cols1-2:\n", a[0:2, 1:3])
print("every other column:  \n", a[:, ::2])
print("last row (negative):", a[-1])

⚠️ **Critical trap — views vs copies.** A *slice* of a NumPy array is a **view**: it shares
memory with the original. Modifying the slice modifies the original! Use `.copy()` when you
want an independent array.

In [ ]:
original = np.array([1, 2, 3, 4, 5])
view = original[1:4]     # a VIEW, shares memory
view[0] = 999
print("original changed!:", original)   # [1 999 3 4 5]  <- surprise!

original2 = np.array([1, 2, 3, 4, 5])
independent = original2[1:4].copy()      # a real copy
independent[0] = 999
print("original2 safe:   ", original2)   # [1 2 3 4 5]

### ✏️ Your Turn 4.1
Given the array below (a small image-like grid), extract:
1. The center 2×2 block
2. The last column
3. Every element in reverse order along both axes (hint: `[::-1, ::-1]`)

In [ ]:
grid = np.array([[1, 2, 3, 4],
                 [5, 6, 7, 8],
                 [9,10,11,12],
                 [13,14,15,16]])
center = None
last_col = None
reversed_grid = None
print(center); print(last_col); print(reversed_grid)

✅ **Solution**
```python
center = grid[1:3, 1:3]        # [[6,7],[10,11]]
last_col = grid[:, -1]         # [4,8,12,16]
reversed_grid = grid[::-1, ::-1]
```

---
## Chapter 5 — Boolean Masking & Fancy Indexing

📖 **Theory.** This is how you *filter* data — the NumPy equivalent of SQL's `WHERE`.
- A **boolean mask** is a same-shaped array of True/False; `arr[mask]` keeps the True spots.
- **Fancy indexing** uses an array of integer positions to pick specific elements.

⚡ Combine conditions with `&` (and), `|` (or), `~` (not) — **never** Python's `and`/`or`
(they raise an error on arrays). Wrap each condition in parentheses.

In [ ]:
data = np.array([12, 45, 7, 88, 23, 61, 34, 9, 77])

mask = data > 30
print("mask:", mask)
print("values > 30:", data[mask])

# combine conditions
print("between 20 and 70:", data[(data >= 20) & (data <= 70)])

# fancy indexing: pick positions 0, 3, and 8
print("picked positions:", data[[0, 3, 8]])

# masks can also assign
data_copy = data.copy()
data_copy[data_copy < 20] = 0     # zero out everything under 20
print("after masked assignment:", data_copy)

### ✏️ Your Turn 5.1
Using `temperatures` below (daily highs in °C):
1. Get all days above 30°C (a heatwave)
2. Count how many days were freezing (≤ 0°C)
3. Create a copy where every negative temperature is set to 0

In [ ]:
temperatures = np.array([15, 32, -3, 28, 35, 0, -5, 22, 31, 8])
heatwave = None
freezing_count = None
clipped = None
print(heatwave, freezing_count, clipped)

✅ **Solution**
```python
heatwave = temperatures[temperatures > 30]
freezing_count = (temperatures <= 0).sum()
clipped = temperatures.copy()
clipped[clipped < 0] = 0
```

---
## Chapter 6 — Vectorized Math & Universal Functions (ufuncs)

📖 **Theory.** Arithmetic on arrays is *elementwise* and vectorized. NumPy also provides
**ufuncs** — fast elementwise functions like `np.sqrt`, `np.exp`, `np.log`, `np.sin`,
`np.abs`, `np.round`. They operate on the whole array at once.

🧠 **Mental model.** If you're about to write a `for` loop over an array to transform each
element, stop — there's almost always a ufunc or vectorized expression instead.

In [ ]:
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])

print("a + b:", a + b)
print("a * b:", a * b)
print("b / a:", b / a)
print("a squared:", a**2)
print("sqrt(b):", np.sqrt(b))
print("exp(a):", np.exp(a).round(2))
print("log(b):", np.log(b).round(3))

# a practical example: convert Celsius to Fahrenheit for a whole array at once
celsius = np.array([0, 20, 37, 100])
fahrenheit = celsius * 9/5 + 32
print("Fahrenheit:", fahrenheit)

### ✏️ Your Turn 6.1
You have investment values. Compute:
1. The percentage return of each vs. the first value (`(v - v[0]) / v[0] * 100`)
2. The natural log of each value (used in finance for log-returns)
3. Round the percentage returns to 1 decimal place

In [ ]:
values = np.array([100, 112, 98, 130, 145])
pct_return = None
log_values = None
pct_rounded = None
print(pct_rounded, log_values)

✅ **Solution**
```python
pct_return = (values - values[0]) / values[0] * 100
log_values = np.log(values)
pct_rounded = np.round(pct_return, 1)
```

---
## Chapter 7 — Broadcasting (the big one)

📖 **Theory.** Broadcasting lets NumPy combine arrays of *different but compatible* shapes
without you manually copying data. The rule, comparing shapes right-to-left: two dimensions
are compatible if they're **equal** or **one of them is 1**. Missing dimensions are treated
as 1.

🧠 **Mental model.** A dimension of size 1 gets "stretched" (virtually, no memory copied) to
match. A `(3,1)` column and a `(1,4)` row broadcast to a full `(3,4)` grid.

This is what makes operations like "subtract the mean of each column" a one-liner.

In [ ]:
# Scalar broadcasts to everything
a = np.array([1, 2, 3])
print("a + 10:", a + 10)

# (3,1) column + (1,3) row -> (3,3) grid
col = np.array([[1], [2], [3]])       # shape (3,1)
row = np.array([[10, 20, 30]])        # shape (1,3)
print("outer sum grid:\n", col + row)

# Real use: normalize each column of a dataset (feature scaling)
data = np.array([[10, 200, 1],
                 [15, 220, 0],
                 [ 9, 180, 1],
                 [20, 260, 0]], dtype=float)
col_mean = data.mean(axis=0)          # shape (3,)
col_std  = data.std(axis=0)           # shape (3,)
normalized = (data - col_mean) / col_std   # (4,3) with (3,) -> broadcasts
print("normalized:\n", normalized.round(2))

⚠️ **Common trap.** `(4,3) - (4,)` fails because the trailing dimensions (3 vs 4) don't
match. You'd need `(4,1)` to subtract a per-*row* value. Always line shapes up right-to-left.

### ✏️ Your Turn 7.1
`sales` is (5 days × 3 products) of units sold. `price` is the per-product price (shape (3,)).
1. Compute revenue per cell (units × price) using broadcasting
2. Compute total revenue per product (sum down the days)
3. Now subtract each *day's* average units (a per-row value) from `sales` — you'll need to
   reshape the row-means to shape (5,1). (hint: `.reshape(-1, 1)`)

In [ ]:
sales = np.array([[10, 5, 8],
                  [12, 4, 9],
                  [ 9, 6, 7],
                  [15, 3, 10],
                  [11, 7, 6]])
price = np.array([900, 25, 45])
revenue = None
revenue_per_product = None
demeaned = None
print(revenue_per_product); print(demeaned)

✅ **Solution**
```python
revenue = sales * price                        # (5,3) * (3,) -> (5,3)
revenue_per_product = revenue.sum(axis=0)      # (3,)
row_means = sales.mean(axis=1).reshape(-1, 1)  # (5,1)
demeaned = sales - row_means                   # (5,3) - (5,1) -> (5,3)
```

---
## Chapter 8 — Aggregations & the `axis` Parameter

📖 **Theory.** Reduce an array to summary numbers: `sum, mean, std, min, max, median,
argmin, argmax, cumsum`. The **`axis`** parameter controls the direction:
- `axis=0` → collapse **down the rows** → one result **per column**
- `axis=1` → collapse **across the columns** → one result **per row**
- no axis → collapse everything to a single number

🧠 **Mental model.** `axis` = the dimension that *disappears*. `axis=0` removes the row
dimension, leaving column summaries.

In [ ]:
m = np.array([[4, 2, 9],
              [1, 7, 3],
              [6, 5, 8]])

print("sum of everything:", m.sum())
print("column sums (axis=0):", m.sum(axis=0))   # one per column
print("row means (axis=1):  ", m.mean(axis=1))  # one per row
print("max of each column:  ", m.max(axis=0))
print("position of overall max:", np.unravel_index(m.argmax(), m.shape))
print("cumulative sum flat:", m.cumsum())

⚡ **Pro tip.** `argmax`/`argmin` give the *index* of the max/min, not the value — perfect
for "which product/day/category was best?" questions. Use `np.unravel_index` to turn a flat
index back into (row, col).

### ✏️ Your Turn 8.1
`scores` is (4 students × 3 exams). Find:
1. Each student's average (one number per student)
2. Each exam's highest score (one number per exam)
3. The index of the student with the best overall average

In [ ]:
scores = np.array([[80, 75, 90],
                   [60, 85, 70],
                   [95, 92, 88],
                   [50, 65, 72]])
student_avg = None
exam_max = None
best_student_idx = None
print(student_avg, exam_max, best_student_idx)

✅ **Solution**
```python
student_avg = scores.mean(axis=1)
exam_max = scores.max(axis=0)
best_student_idx = scores.mean(axis=1).argmax()
```

---
## Chapter 9 — Reshaping, Stacking & Splitting

📖 **Theory.** Change an array's layout without changing its data:
- `.reshape(new_shape)` — same data, new dimensions (use `-1` to auto-infer one dimension)
- `.ravel()` / `.flatten()` — collapse to 1D
- `.T` — transpose (swap rows/columns)
- `np.vstack` / `np.hstack` / `np.concatenate` — join arrays
- `np.split` / `np.hsplit` / `np.vsplit` — break arrays apart

In [ ]:
a = np.arange(12)
print("original:", a)
reshaped = a.reshape(3, 4)
print("reshaped 3x4:\n", reshaped)
print("auto-infer with -1 (reshape 2,-1):\n", a.reshape(2, -1))
print("transpose:\n", reshaped.T)
print("flattened:", reshaped.ravel())

# stacking
x = np.array([1, 2, 3])
y = np.array([4, 5, 6])
print("vstack:\n", np.vstack([x, y]))
print("hstack:", np.hstack([x, y]))

⚠️ **Common trap.** `reshape` requires the total number of elements to stay the same. A
12-element array can become (3,4) or (2,6) but not (5,3). The `-1` trick lets NumPy compute
one dimension for you so you don't have to do the arithmetic.

### ✏️ Your Turn 9.1
1. Turn `np.arange(24)` into a 4×6 array
2. Transpose it to 6×4
3. Stack two 1D arrays `[1,2,3]` and `[4,5,6]` vertically, then flatten the result back to 1D

In [ ]:
arr = np.arange(24)
grid = None
transposed = None
stacked_flat = None
print(grid); print(transposed); print(stacked_flat)

✅ **Solution**
```python
grid = arr.reshape(4, 6)
transposed = grid.T
stacked_flat = np.vstack([[1,2,3],[4,5,6]]).ravel()
```

---
## Chapter 10 — Sorting, Searching & Set Operations

📖 **Theory.** Useful everyday tools:
- `np.sort(a)` returns a sorted copy; `a.argsort()` returns the indices that would sort it
- `np.where(condition, x, y)` — vectorized if/else
- `np.unique(a)` — sorted unique values (optionally with counts)
- `np.in1d` / `np.intersect1d` / `np.union1d` — set operations

In [ ]:
a = np.array([3, 1, 4, 1, 5, 9, 2, 6])
print("sorted:", np.sort(a))
print("argsort (indices):", a.argsort())

# np.where as vectorized if/else: label pass/fail
scores = np.array([55, 80, 45, 90, 70])
labels = np.where(scores >= 60, "pass", "fail")
print("labels:", labels)

# unique with counts
colors = np.array(["red","blue","red","green","blue","red"])
vals, counts = np.unique(colors, return_counts=True)
print("unique:", vals, "counts:", counts)

# set operations
setA = np.array([1,2,3,4])
setB = np.array([3,4,5,6])
print("intersection:", np.intersect1d(setA, setB))
print("union:", np.union1d(setA, setB))

### ✏️ Your Turn 10.1
Given `purchases` (product names) and `amounts`:
1. Find the unique products and how many times each was purchased
2. Use `np.where` to label each amount as "big" (≥100) or "small"
3. Get the amounts sorted from highest to lowest (hint: sort then reverse with `[::-1]`)

In [ ]:
purchases = np.array(["pen","book","pen","laptop","book","pen"])
amounts = np.array([5, 20, 8, 900, 25, 6])
unique_products, product_counts = None, None
size_labels = None
amounts_desc = None
print(unique_products, product_counts, size_labels, amounts_desc)

✅ **Solution**
```python
unique_products, product_counts = np.unique(purchases, return_counts=True)
size_labels = np.where(amounts >= 100, "big", "small")
amounts_desc = np.sort(amounts)[::-1]
```

---
## Chapter 11 — Linear Algebra Essentials

📖 **Theory.** NumPy powers ML math. Key operations:
- `A @ B` or `np.dot(A, B)` — matrix multiplication
- `A.T` — transpose
- `np.linalg.inv(A)` — matrix inverse
- `np.linalg.solve(A, b)` — solve linear system `Ax = b` (better than inverting)
- `np.linalg.norm(v)` — vector length/magnitude

⚡ Matrix multiply `@` is NOT elementwise `*`. `@` follows linear-algebra rules: (m×n) @ (n×p) = (m×p).

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print("elementwise A * B:\n", A * B)
print("matrix product A @ B:\n", A @ B)
print("transpose:\n", A.T)

# Solve a system:  2x + y = 5 ,  x + 3y = 10
coeffs = np.array([[2, 1], [1, 3]])
rhs = np.array([5, 10])
solution = np.linalg.solve(coeffs, rhs)
print("solution [x, y]:", solution)

# Vector magnitude (Euclidean length)
v = np.array([3, 4])
print("norm of [3,4]:", np.linalg.norm(v))   # 5.0

### ✏️ Your Turn 11.1
1. Multiply matrix `M = [[2,0],[1,3]]` by vector `v = [4, 5]` (result should be shape (2,))
2. Compute the cosine similarity between `u=[1,2,3]` and `w=[2,4,6]`
   (formula: `(u @ w) / (norm(u) * norm(w))`). What value do you expect for parallel vectors?

In [ ]:
M = np.array([[2,0],[1,3]])
v = np.array([4, 5])
Mv = None
u = np.array([1,2,3]); w = np.array([2,4,6])
cos_sim = None
print(Mv, cos_sim)

✅ **Solution**
```python
Mv = M @ v
cos_sim = (u @ w) / (np.linalg.norm(u) * np.linalg.norm(w))
# cos_sim = 1.0 because w is exactly parallel to u (same direction)
```

---
## Chapter 12 — Random Numbers & Simulation

📖 **Theory.** `np.random` generates random data for simulation, sampling, and ML.
- `np.random.seed(n)` — makes randomness reproducible (same numbers every run)
- `np.random.rand(shape)` — uniform [0,1); `randn` — standard normal
- `np.random.randint(low, high, size)` — random integers
- `np.random.choice(options, size, p=probabilities)` — sample from choices

⚡ Always set a seed in tutorials/tests so results are reproducible.

In [ ]:
np.random.seed(0)
print("uniform:", np.random.rand(3).round(3))
print("normal: ", np.random.randn(3).round(3))
print("integers:", np.random.randint(1, 7, 5))      # like rolling a die 5 times
print("weighted choice:", np.random.choice(["A","B","C"], 5, p=[0.7, 0.2, 0.1]))

# Simulation: estimate pi with random points in a unit square (Monte Carlo)
np.random.seed(1)
n = 100_000
points = np.random.rand(n, 2)
inside_circle = (points[:,0]**2 + points[:,1]**2) <= 1
pi_estimate = 4 * inside_circle.mean()
print(f"Monte Carlo pi estimate: {pi_estimate:.4f}")

### ✏️ Your Turn 12.1
Simulate rolling two dice 10,000 times:
1. Generate two arrays of 10,000 random integers each from 1–6
2. Compute the sum of the two dice for each roll
3. What fraction of rolls summed to exactly 7? (the most likely sum)

In [ ]:
np.random.seed(7)
die1 = None
die2 = None
roll_sums = None
fraction_seven = None
print(fraction_seven)

✅ **Solution**
```python
die1 = np.random.randint(1, 7, 10000)
die2 = np.random.randint(1, 7, 10000)
roll_sums = die1 + die2
fraction_seven = (roll_sums == 7).mean()   # ~0.167
```

---
## 🏆 Chapter 13 — Capstone Project: Student Performance Analysis

Time to combine everything. We'll load a real CSV of student scores and answer questions
using only NumPy — no Pandas yet (that's the next lab!).

**Dataset:** `datasets/student_scores.csv` — 30 students × 5 subjects (math, science,
english, history, art).

In [ ]:
# Load the CSV (skip the header row) into a NumPy array
scores = np.loadtxt("datasets/student_scores.csv", delimiter=",", skiprows=1)
subjects = ["math", "science", "english", "history", "art"]
print("shape:", scores.shape)
print("first 3 students:\n", scores[:3])

### ✏️ Capstone Tasks
Answer each using NumPy. Work through them before checking the solutions.

1. **Class average per subject** — the mean score in each subject
2. **Top student** — the index and average of the student with the highest overall average
3. **Hardest subject** — which subject had the lowest class average
4. **Honor roll** — how many students have an *average* score ≥ 80
5. **Curve the grades** — add 5 points to every score, but cap all scores at 100
6. **Pass/fail matrix** — a boolean array marking which scores are ≥ 60

In [ ]:
# 1. Class average per subject
subject_avgs = None

# 2. Top student
student_avgs = None
top_student_idx = None
top_student_avg = None

# 3. Hardest subject
hardest_subject = None

# 4. Honor roll count
honor_roll_count = None

# 5. Curve (add 5, cap at 100)
curved = None

# 6. Pass/fail matrix
pass_matrix = None

print("Subject averages:", None)
print("Top student:", None)
print("Hardest subject:", None)
print("Honor roll size:", None)

✅ **Capstone Solution**
```python
# 1
subject_avgs = scores.mean(axis=0)

# 2
student_avgs = scores.mean(axis=1)
top_student_idx = student_avgs.argmax()
top_student_avg = student_avgs.max()

# 3
hardest_subject = subjects[subject_avgs.argmin()]

# 4
honor_roll_count = (student_avgs >= 80).sum()

# 5
curved = np.minimum(scores + 5, 100)

# 6
pass_matrix = scores >= 60

print("Subject averages:", dict(zip(subjects, subject_avgs.round(1))))
print(f"Top student: #{top_student_idx} with {top_student_avg:.1f}")
print("Hardest subject:", hardest_subject)
print("Honor roll size:", honor_roll_count)
```

🎉 **Congratulations!** You've gone from zero to a working command of NumPy: arrays,
indexing, masking, vectorization, broadcasting, aggregations, reshaping, linear algebra, and
simulation. These same ideas power Pandas, PyTorch, and every ML library.

**Next:** open `Pandas_Zero_to_Master.ipynb` in this folder — Pandas is built directly on
top of everything you just learned.

---
### 📌 Function Quick-Reference (everything used in this lab)
**Creation:** `array, arange, linspace, zeros, ones, full, eye, loadtxt`
**Random:** `seed, rand, randn, randint, choice`
**Inspection:** `.shape, .ndim, .size, .dtype, .astype`
**Indexing:** `[i,j]`, slicing `start:stop:step`, boolean masks, fancy indexing, `.copy()`
**Math/ufuncs:** `+ - * / **`, `sqrt, exp, log, abs, round, minimum, maximum`
**Aggregation:** `sum, mean, std, min, max, median, argmin, argmax, cumsum` (+ `axis`)
**Reshape/join:** `reshape, ravel, flatten, .T, vstack, hstack, concatenate, split`
**Search/sort/set:** `sort, argsort, where, unique, intersect1d, union1d`
**Linear algebra:** `@, dot, linalg.inv, linalg.solve, linalg.norm, unravel_index`
